In [1]:
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
from IPython.display import display

import matplotlib.pyplot as plt
import pandas as pd


def round_half_up(values, ndigits=2):
    quant = Decimal("1").scaleb(-ndigits)

    def _round(x):
        if pd.isna(x):
            return x
        return float(Decimal(str(x)).quantize(quant, rounding=ROUND_HALF_UP))

    return values.map(_round)

RESULTS_DIR = Path(".")
COLS = ["run_name", "epoch", "category", "n_examples", "accuracy_mean", "accuracy_std"]
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)


def load_accuracy_by_category(name, csv_path, exclude_categories=None):
    if not csv_path.exists():
        print(f"skip {name}: missing {csv_path.name}")
        return None

    df = pd.read_csv(csv_path)
    df = df[(df["testset"].isin(["dative_accusative_pairs", "dative_accusative_pairs_shifted"])) & (df["n_seeds"].isin([1, 4, 5]))]
    df = df[COLS].copy()
    if df.empty:
        print(f"skip {name}: no matching rows")
        return None

    df = df.sort_values(["category", "epoch"])
    df["accuracy_mean"] = round_half_up(df["accuracy_mean"] * 100)
    df["accuracy_std"] = round_half_up(df["accuracy_std"] * 100)
    df["run_name"] = name

    if exclude_categories:
        df = df[~df["category"].isin(exclude_categories)]
        if df.empty:
            print(f"skip {name}: no categories left after exclude_categories={exclude_categories}")
            return None
    return df


def display_tables_by_epoch(frames, epochs=range(-1,6)):
    if not frames:
        print("no model tables to display")
        return
    all_df = pd.concat(frames, ignore_index=True)
    for epoch in epochs:
        sub = all_df[all_df["epoch"] == epoch]
        if sub.empty:
            continue
        table = (
            sub.pivot(index="run_name", columns="category", values="accuracy_mean")
            .sort_index()
            .reset_index()
            .rename(columns={"run_name": "model"})
        )
        table.columns.name = None
        print(f"\n## epoch {epoch}")
        display(
            table.style.format(precision=2).set_properties(
                subset=["model"],
                **{"white-space": "nowrap", "text-align": "left"},
            )
        )


def plot_accuracy_by_category(name, csv_path, out_path=None, exclude_categories=None):
    plot_df = load_accuracy_by_category(name, csv_path, exclude_categories=exclude_categories)
    if plot_df is None:
        return None

    y_min = plot_df["accuracy_mean"].min() - 3

    fig, ax = plt.subplots(figsize=(8, 5))
    for category, group in plot_df.groupby("category", sort=True):
        ax.plot(group["epoch"], group["accuracy_mean"], marker="o", label=str(category))
        ax.fill_between(
            group["epoch"],
            group["accuracy_mean"] - group["accuracy_std"],
            group["accuracy_mean"] + group["accuracy_std"],
            alpha=0.2,
        )

    ax.set_xlabel("epoch")
    ax.set_ylabel("accuracy (%)")
    ax.set_ylim(bottom=y_min)
    ax.set_title(f"{name} — dative_accusative_pairs")
    ax.legend(title="category", bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    if out_path is None:
        out_path = csv_path.parent / f"{name}_accuracy.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return plot_df


In [ ]:
for folder in sorted(p for p in RESULTS_DIR.iterdir() if p.is_dir()):
    name = folder.name
    csv_path = folder / f"{name}_aggregate_by_category.csv"
    out_path = f"{name}_accuracy.png"
    # print(f"plotting {name} at {out_path}")
    plot_accuracy_by_category(name, csv_path, out_path)

In [2]:
frames = []
for folder in sorted(p for p in RESULTS_DIR.iterdir() if p.is_dir()):
    name = folder.name
    csv_path = folder / f"{name}_aggregate_by_category.csv"
    out_path = f"{name}_accuracy.png"
    df = plot_accuracy_by_category(name, csv_path, out_path)
    if df is not None:
        frames.append(df)
display_tables_by_epoch(frames)


## epoch 1


,model,ALL,Ablative_TR,Instrumental_TR,Locative_TR,Mixed_TR,NaN_prep,dative-case,no-case,other-case
0,char-de_phase_2_de_nllb_regular,77.52,86.60,94.51,76.52,84.36,60.48,26.77,30.60,33.11
1,char-de_phase_2_de_nllb_shifted,77.50,86.80,95.18,76.67,84.03,58.62,26.22,29.85,31.26
2,char-en-de_phase_2_regular_de_nllb_regular,82.50,90.02,97.42,80.69,92.76,71.92,30.16,41.94,31.48
3,char-en-de_phase_2_regular_de_nllb_reset_embeddings,80.65,88.68,96.57,78.82,90.75,69.16,26.30,37.31,29.95
4,char-en-de_phase_2_regular_de_nllb_reset_embeddings_shifted,80.96,89.14,96.87,78.89,91.51,70.12,27.01,37.31,29.62
5,char-en-de_phase_2_regular_de_nllb_shifted,81.88,89.34,97.03,80.29,91.62,69.82,29.69,42.69,30.60
6,char-en-de_phase_2_shuffled_de_nllb_regular,82.36,89.78,97.35,80.53,92.96,70.66,29.92,42.09,30.71
7,char-tr-de_phase_2_regular_de_nllb_regular,82.16,89.92,97.42,80.20,92.51,71.68,28.82,43.88,30.27
8,char-tr-de_phase_2_regular_de_nllb_reset_embeddings,81.07,88.93,96.55,79.14,91.81,68.98,27.24,39.10,28.96
9,char-tr-de_phase_2_regular_de_nllb_reset_embeddings_shifted,80.99,89.46,96.82,78.80,91.45,68.74,27.80,40.90,30.82



## epoch 3


,model,ALL,Ablative_TR,Instrumental_TR,Locative_TR,Mixed_TR,NaN_prep,dative-case,no-case,other-case
0,char-de_phase_2_de_nllb_regular,82.60,90.05,97.55,80.65,93.36,70.78,29.92,44.03,31.69
1,char-de_phase_2_de_nllb_shifted,83.01,89.83,97.49,81.27,93.77,71.26,30.79,44.93,32.02
2,char-en-de_phase_2_regular_de_nllb_regular,84.87,92.12,97.94,82.91,94.32,75.81,38.43,54.33,37.60
3,char-en-de_phase_2_regular_de_nllb_reset_embeddings,83.82,91.73,98.15,80.95,94.68,75.03,37.09,54.03,34.97
4,char-en-de_phase_2_regular_de_nllb_reset_embeddings_shifted,84.18,91.86,97.94,81.73,95.12,73.89,36.14,52.84,33.99
5,char-en-de_phase_2_regular_de_nllb_shifted,84.55,91.56,97.69,82.48,94.76,74.85,36.22,53.73,37.81
6,char-en-de_phase_2_shuffled_de_nllb_regular,84.61,91.34,97.76,82.57,94.76,75.27,36.77,55.07,37.49
7,char-tr-de_phase_2_regular_de_nllb_regular,84.82,92.30,97.78,82.78,95.10,75.09,36.22,51.79,36.72
8,char-tr-de_phase_2_regular_de_nllb_reset_embeddings,84.58,92.01,97.92,82.15,95.09,74.79,37.48,57.01,36.83
9,char-tr-de_phase_2_regular_de_nllb_reset_embeddings_shifted,83.95,91.81,97.71,81.22,95.16,73.59,36.22,54.48,35.63



## epoch 5


,model,ALL,Ablative_TR,Instrumental_TR,Locative_TR,Mixed_TR,NaN_prep,dative-case,no-case,other-case
0,char-de_phase_2_de_nllb_regular,83.76,91.00,97.78,81.82,94.57,72.40,31.34,48.36,34.43
1,char-de_phase_2_de_nllb_shifted,83.90,90.96,97.58,81.92,94.97,72.69,32.20,49.25,33.66
2,char-en-de_phase_1_regular,46.12,53.76,49.19,43.53,42.29,57.90,50.24,48.81,56.17
3,char-en-de_phase_1_shuffled,46.81,51.83,52.57,44.07,46.35,59.22,43.39,31.19,54.43
4,char-en-de_phase_2_regular_de_nllb_regular,85.66,92.93,98.08,83.48,95.25,77.25,40.87,56.27,41.20
5,char-en-de_phase_2_regular_de_nllb_reset_embeddings,84.69,92.32,98.35,82.09,95.30,75.45,38.43,54.63,37.05
6,char-en-de_phase_2_regular_de_nllb_reset_embeddings_shifted,85.06,92.74,98.10,82.75,95.78,74.31,37.87,53.88,36.28
7,char-en-de_phase_2_regular_de_nllb_shifted,85.53,92.55,97.87,83.43,95.59,76.53,40.08,54.63,39.78
8,char-en-de_phase_2_shuffled_de_nllb_regular,85.45,92.44,98.26,83.12,95.52,76.95,39.76,57.01,40.98
9,char-tr-de_phase_1_regular,46.36,49.90,47.54,47.83,42.63,51.98,30.94,25.52,42.95
